# Exchange · Save and read COMPAS JSON
**CAS Computational Form-finding · Block Research Group · ETH Zürich**

**Why this is needed:** a Python session disappears when its kernel closes.
Saving geometry and its units lets you reopen a design or share it with someone
else. We round-trip just a **box**, a **point** and a small metadata dictionary.
No previous notebook needs to have run.

COMPAS JSON restores COMPAS object types. It is **not IFC**; IFC introduces an AEC
schema and building semantics in a later lesson. **Predict:** does saving a box's
name alone preserve its dimensions?


## 1. Start Python
Run all cells in order with **Run → Run All Cells**, or step through with **Shift+Enter**.
JupyterLite supplies NumPy and SciPy. On desktop Jupyter, install them with
`%pip install -q numpy scipy` if an import below reports a missing package.
The pinned COMPAS packages below are the course's tested versions; `--no-deps`
avoids installing a desktop notebook server in browser Python.


In [ ]:
%pip install -q compas==2.15.1 pythreejs==2.4.2
%pip install -q compas_notebook==0.11.0 --no-deps


In [ ]:
# Enable widgets on Colab; browser JupyterLite and desktop Jupyter need no change.
try:
    from google.colab import output
except ImportError:
    pass
else:
    output.enable_custom_widget_manager()


In [ ]:
from pathlib import Path
import compas
from compas.geometry import Box, Point
from compas_notebook import Viewer
from compas_notebook.config import Config

box = Box(2.0, 1.0, 1.5, name="Study box")
point = Point(1, 0, 0)
session = {"schema_version": "cas-geometry-1", "length_unit": "m",
           "box": box, "reference_point": point,
           "notes": "A minimal reusable geometry study"}


## Write a file, then read it into new objects
`compas.json_dump` writes the session. `compas.json_load` reconstructs it.
The file is created in this notebook's working directory; in JupyterLite,
download it from the file browser to keep a copy outside this browser.


In [ ]:
json_path = Path("box-study.compas.json")
compas.json_dump(session, json_path, pretty=True)
print(f"Saved {json_path}: {json_path.stat().st_size} bytes")
restored = compas.json_load(json_path)
restored_box = restored["box"]
print(type(restored_box).__name__, restored_box.name, restored_box.volume, "m³")


## Check that meaning survived
Check object type, dimensions, name, point coordinates and units. File size alone
does not tell us whether the right geometry was saved. The JSON text includes
COMPAS type information; use the COMPAS decoder to restore the geometric objects.


In [ ]:
assert isinstance(restored_box, Box)
assert isinstance(restored["reference_point"], Point)
assert restored_box.name == box.name
assert [restored_box.xsize, restored_box.ysize, restored_box.zsize] == [box.xsize, box.ysize, box.zsize]
assert abs(restored_box.volume - box.volume) < 1e-9
assert list(restored["reference_point"]) == list(point)
assert restored["length_unit"] == session["length_unit"]
assert restored["schema_version"] == session["schema_version"]
print("Round trip preserved the objects and their metadata.")

config = Config()
config.view.width = 700
config.view.height = 360
config.view.show_grid = False

config.view.camera.position = [4, -6, 3]
config.view.camera.target = [0, 0, 0]
view = Viewer(config=config)
view.scene.add(restored_box.to_mesh(), color="#5552bb", show_edges=True)
view.show()


## Predict → change → check
Starting result: decoded object type **Box**, name **Study box**, volume **3 m³**.

1. Change one box dimension, predict the volume, then save and read it again.
2. Add your name to the session dictionary and check that it survives the round trip.
3. Ask a colleague to read your file and report its units and box dimensions.
4. Explain what additional information a building model would need beyond this box.

For a later mini-project, save a parameter study with geometry, units, design
question and one numerical check. Reopen it in a fresh kernel before sharing it.


In [ ]:
# Safe experiment: JSON strings also round-trip without creating another file.
text = compas.json_dumps(session, pretty=True)
trial = compas.json_loads(text)
assert isinstance(trial["box"], Box)
assert abs(trial["box"].volume - box.volume) < 1e-9
print("In-memory round trip:", len(text), "characters")


## Sources
Original course exercise informed by [Seoul 2025 JSON examples](https://github.com/BRG-teaching/seoul_2025/tree/b641b732a3e337eb3a19d7a91048946820695e7a/05_compas)
and [UCL 2024 data exchange](https://github.com/BRG-teaching/UCL_2024/tree/6bec117f799e59be39529aa2bd6f3aa203d4ba4d).
API: [COMPAS data](https://compas.dev/compas/2.15.1/api/compas.data.html).
Viewer pattern: [COMPAS JupyterLite](https://github.com/compas-dev/compas-jupyterlite/tree/d9df5aef10685c80507236554900d72b86504d6a).

Save with **Ctrl+S** (or **⌘S**), then use **File → Save and Export Notebook As → Notebook** to download your work. Browser files are local to this browser; export a copy before moving machines.
